In [ ]:
import numpy as np
from datasets import load_dataset

# ============================================================================
# 🌟 AI 코딩 튜토리얼: 스피치 인식 데이터셋 탐험! 🎧
# 🚀 데이터셋명: JaepaX/korean_dataset
# 📝 데이터셋 의미: 한국어로 녹음된 음성(Audio)과 해당 음성에 대한 정확한 텍스트(Transcription) 쌍으로 이루어진 대용량 데이터셋입니다.
# 💡 목표: 이 스크립트는 초급자가 ASR(자동 음성 인식) 데이터를 어떻게 구조적으로 이해하고,
#       실제 모델 학습에 들어가기 전에 데이터를 안전하게 '탐험'하는 방법을 보여줍니다.
# ============================================================================

# ✨ 우리가 만날 데이터셋 정보
DATASET_NAME = "JaepaX/korean_dataset"
SAMPLE_COUNT = 5  # 데이터셋 전체를 사용하지 않고, 상위 5개만 살펴볼 거예요!

print("--------------------------------------------------------------------------")
print("✅ 튜토리얼 시작! 준비되셨나요? 😎 이 데이터셋은 '음성' 데이터를 다룹니다.")
print("--------------------------------------------------------------------------")

dataset = None
sample_data_list = []

# ----------------------------------------------------------------------------
# 🛠️ 1단계: 데이터셋 로딩 전략 짜기 (스트리밍 vs. 일반 로딩)
# ----------------------------------------------------------------------------
print("\n[STEP 1/3] 데이터셋 로딩을 시도합니다... (스트리밍 모드 우선)")

try:
    # 💡 스트리밍 방식 (streaming=True)으로 시도합니다. 메모리 효율이 뛰어나지만,
    #    때때로 환경에 따라 로딩에 실패할 수 있어요.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✨ 성공! 데이터를 스트리밍 모드로 로드했습니다. (메모리 친화적)")

except Exception as e:
    print(f"⚠️ 스트리밍 로딩 실패 또는 제한사항 감지: {e}")
    print("➡️ 대안 로딩 모드로 전환하여 작업을 계속하겠습니다.")
    try:
        # 🚨 스트리밍이 실패하면, 작은 샘플만 일반 모드로 로드하여 안정성을 확보합니다.
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✅ 대체 로딩 성공! 소규모 샘플로 작업을 진행합니다.")
    except Exception as e_fallback:
        print(f"❌ 모든 로딩 시도 실패. 오류: {e_fallback}. 스크립트를 종료합니다.")
        exit()


# ----------------------------------------------------------------------------
# 🎈 2단계: 데이터 샘플 추출 및 검토 (전체 루프 대신 take() 사용!)
# ----------------------------------------------------------------------------
print(f"\n[STEP 2/3] 상위 {SAMPLE_COUNT}개의 샘플만 추출하여 탐색합니다.")

# 🔄 데이터셋이 스트리밍인지 일반 데이터셋인지 판별하고,
#    필수 요구사항에 따라 .take() 패턴을 사용하여 샘플러를 만듭니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    sample_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)
    # list()로 변환하면, Iterator 패턴을 따르면서도 샘플을 확보하기 좋습니다.
    sample_data_list = list(dataset.select(range(SAMPLE_COUNT)))
    sample_iterator = iter(sample_data_list)

sample_count = 0

# 💡 추출된 샘플들을 순회하며 내용을 확인합니다. (for-each 루프를 사용)
print("=" * 70)
print("🔎 추출된 샘플별 데이터 구조 분석 결과:")
print("=" * 70)

for i, sample in enumerate(sample_iterator):
    sample_count += 1
    print(f"\n=== ✨ [Sample {i+1}/{SAMPLE_COUNT}] 구조 분석 시작 === 📝")

    # 1. 전사 텍스트 분석 (Transcription Analysis)
    transcription = sample["transcription"]
    print(f"  📚 텍스트(Transcription): '{transcription[:40]}...' (길이: {len(transcription)}자)")

    # 2. 오디오 특징 분석 (Audio Feature Inspection)
    # ❗ 핵심: 이 데이터셋의 가장 중요한 정보는 'audio' 필드에 들어있어요!
    audio_data = sample["audio"]
    
    # 오디오 데이터의 샘플링 레이트와 배열 데이터를 확인합니다.
    sampling_rate = audio_data['sampling_rate']
    audio_array = audio_data['array']
    
    # 오디오 배열의 크기를 확인하여 '정량적 분석'을 수행합니다.
    # audio_array.ndim을 사용하여 차원 정보를 알려줍니다.
    print(f"  🔊 오디오 데이터 (Audio):")
    print(f"    - 샘플링 레이트 (Sampling Rate): {sampling_rate} Hz")
    print(f"    - 데이터 형식: NumPy Array ({audio_array.dtype})")
    print(f"    - 크기 (Shape): {audio_array.shape} (총 샘플 수: {audio_array.shape[0]}개)")

    # 3. 창의적 실습 예시: 데이터 불일치 예측 (Mismatch Prediction Simulation)
    # 💡 실제 ASR에서 중요한 것은 오디오와 텍스트의 일관성입니다.
    #    우리는 단순히 텍스트 길이에 비례하여 오디오가 어느 정도 길어야 할지 가상의 지표를 만들어 볼 수 있어요.
    
    # (가상의 계산: 1글자당 평균 0.2초의 오디오가 필요하다고 가정)
    avg_time_per_char = 0.2
    estimated_min_duration = len(transcription) * avg_time_per_char
    
    # 실제 오디오 지속 시간 계산 (초 단위)
    actual_duration = audio_array.shape[0] / sampling_rate
    
    print("  📏 [✨ Analysis] 예측 대비 지속 시간 비교:")
    print(f"    - 텍스트 기반 최소 예상 지속 시간: 약 {estimated_min_duration:.2f} 초")
    print(f"    - 실제 오디오 지속 시간: 약 {actual_duration:.2f} 초")
    
    # 💡 분석 결론: 두 값이 비슷할수록 녹음 품질이 좋거나 길이가 일치한다고 추정할 수 있습니다!
    if abs(actual_duration - estimated_min_duration) / max(1, actual_duration, estimated_min_duration) < 0.2:
        print("    -> 🎯 분석 결과: 텍스트와 오디오의 길이 비율이 양호합니다. (Good Match!)")
    else:
        print("    -> 🧐 분석 결과: 길이가 크게 차이 납니다. (Attention Needed!)")


# ----------------------------------------------------------------------------
# 🎁 3단계: 최종 정리 및 요약 (정량적 분석 결과 출력)
# ----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("🎉 데이터 탐험을 성공적으로 마쳤습니다! 🎉")
print("=" * 70)
print(f"🚀 총 {sample_count}개의 샘플을 성공적으로 탐색했습니다.")
print(f"🔑 주요 데이터 특징:")
print(f"   - 오디오 데이터는 {dataset.features['audio']['dtype'].__name__} 타입으로 저장되며,")
print(f"     샘플링 레이트는 {dataset.features['audio']['sampling_rate']} Hz가 표준입니다.")
print(f"   - 이 데이터셋은 단순히 텍스트와 오디오를 매칭시키는 '쌍(Pair)' 데이터셋입니다.")
print("\n🌟 튜터 코멘트:")
print("   축하합니다! 여러분은 AI 모델의 가장 기초가 되는 '데이터 구조 이해' 단계까지 완벽하게 마스터했습니다.")
print("   다음 단계는 이 데이터를 가지고 실제 음성 모델 학습을 시도하는 것입니다! 😃")